<a href="https://colab.research.google.com/github/Govindu-Sathruwan/CRM-CM-CW1/blob/main/CRM_%26_CM_CW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 Data Preparation and Feature Engineering

## 1.1 Ticker selection and data acquisition

In [4]:
# Import all required libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Part 1: Extract S&P 500 tickers from Wikipedia using Pandas

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_tickers = table[0]

tickers = sp500_tickers['Symbol'].tolist()
tickers = [t.replace('.', '-') for t in tickers]

print('Total tickers extracted:', len(tickers))
sp500_tickers.head(10)

Total tickers extracted: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
5,ADBE,Adobe Inc.,Information Technology,Application Software,"San Jose, California",1997-05-05,796343,1982
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors,"Santa Clara, California",2017-03-20,2488,1969
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders,"Arlington, Virginia",1998-10-02,874761,1981
8,AFL,Aflac,Financials,Life & Health Insurance,"Columbus, Georgia",1999-05-28,4977,1955
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,"Santa Clara, California",2000-06-05,1090872,1999


In [6]:
# Load Daily Market Data via yfinance
start = '2021-01-01'
end = '2025-01-01'

# Include the ^GSPC benchmark with tickers
tickers_with_benchmark = tickers + ['^GSPC']

# Download Open, High, Low, Close, Adj Close, and Volume
market_data = yf.download(tickers_with_benchmark, start=start, end=end, auto_adjust=False)

print('Market Data:', market_data.shape)
display(market_data.head())

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['Q', 'VYLR', 'HONA', 'SNDK', 'FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Market Data: (1005, 3024)


Price        Adj Close                                                \
Ticker               A        AAPL       ABBV        ABNB        ABT   
Date                                                                   
2021-01-04  114.009125  125.632492  84.219147  139.149994  97.269699   
2021-01-05  114.941254  127.185806  85.090027  148.300003  98.473183   
2021-01-06  118.093231  122.904518  84.354965  142.770004  98.268173   
2021-01-07  121.235596  127.098434  85.257790  151.270004  99.222069   
2021-01-08  122.100471  128.195435  85.705208  149.770004  99.498421   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2021-01-04  33.186359  232.029343  485.339996  132.622604  42.507668  ...   
2021-01-05  33.319485  233.350296  485.690002  134.993164  43.334469  ...   
2021-01-06  34.783871  235.901611  466.309998  135.601685  45.090355  ...   
2021-01-07  34.460564  238.127304  477.739990  141.332748  44.689735  ...   
2021-01-08  34.650742  238.995819  485.100006  142.359085  44.809055  ...   

Price        Volume                                                          \
Ticker         WYNN      XEL       XOM      XYL       XYZ      YUM      ZBH   
Date                                                                          
2021-01-04  3596800  3197100  27764700   814700   9838400  2058200  1265870   
2021-01-05  1814600  1976800  44035100  1005200   9699400  1475300   998894   
2021-01-06  2433700  2667000  36484800  1015300  12003500  1352700  1424799   
2021-01-07  3846700  2630400  29528100  1539300  10214200  2222900  1237648   
2021-01-08  2812400  2221500  28628200  1632200   9102100  1252200  1259072   

Price                                    
Ticker        ZBRA      ZTS       ^GSPC  
Date                                     
2021-01-04  432600  1805900  5015000000  
2021-01-05  374700  1358600  4591020000  
2021-01-06  419500  2139800  6064110000  
2021-01-07  379900  1710700  5099160000  
2021-01-08  348500  1475900  4773040000  

[5 rows x 3024 columns]

In [7]:
# Transpose DataFrame: rows = tickers, columns = dates
market_data_1 = market_data.T

print('Shape after transpose:', market_data_1.shape)
market_data_1

Shape after transpose: (3024, 1005)


Date                2021-01-04    2021-01-05    2021-01-06    2021-01-07  \
Price     Ticker                                                           
Adj Close A       1.140091e+02  1.149413e+02  1.180932e+02  1.212356e+02   
          AAPL    1.256325e+02  1.271858e+02  1.229045e+02  1.270984e+02   
          ABBV    8.421915e+01  8.509003e+01  8.435497e+01  8.525779e+01   
          ABNB    1.391500e+02  1.483000e+02  1.427700e+02  1.512700e+02   
          ABT     9.726970e+01  9.847318e+01  9.826817e+01  9.922207e+01   
...                        ...           ...           ...           ...   
Volume    YUM     2.058200e+06  1.475300e+06  1.352700e+06  2.222900e+06   
          ZBH     1.265870e+06  9.988940e+05  1.424799e+06  1.237648e+06   
          ZBRA    4.326000e+05  3.747000e+05  4.195000e+05  3.799000e+05   
          ZTS     1.805900e+06  1.358600e+06  2.139800e+06  1.710700e+06   
          ^GSPC   5.015000e+09  4.591020e+09  6.064110e+09  5.099160e+09   

Date                2021-01-08    2021-01-11    2021-01-12    2021-01-13  \
Price     Ticker                                                           
Adj Close A       1.221005e+02  1.231479e+02  1.220140e+02  1.220236e+02   
          AAPL    1.281954e+02  1.252150e+02  1.250403e+02  1.270693e+02   
          ABBV    8.570521e+01  8.710342e+01  8.771062e+01  8.984386e+01   
          ABNB    1.497700e+02  1.481300e+02  1.608000e+02  1.699900e+02   
          ABT     9.949842e+01  9.881195e+01  9.702901e+01  9.997981e+01   
...                        ...           ...           ...           ...   
Volume    YUM     1.252200e+06  1.329100e+06  1.182600e+06  1.538800e+06   
          ZBH     1.259072e+06  1.237751e+06  1.212310e+06  1.725662e+06   
          ZBRA    3.485000e+05  3.126000e+05  3.131000e+05  2.147000e+05   
          ZTS     1.475900e+06  1.117000e+06  1.304200e+06  2.603100e+06   
          ^GSPC   4.773040e+09  4.465430e+09  4.994950e+09  4.602510e+09   

Date                2021-01-14    2021-01-15  ...    2024-12-17    2024-12-18  \
Price     Ticker                              ...                               
Adj Close A       1.211107e+02  1.212836e+02  ...  1.343538e+02  1.309570e+02   
          AAPL    1.251471e+02  1.234288e+02  ...  2.516537e+02  2.462629e+02   
          ABBV    9.023185e+01  8.933463e+01  ...  1.645307e+02  1.623167e+02   
          ABNB    1.805000e+02  1.692700e+02  ...  1.322400e+02  1.275400e+02   
          ABT     9.849398e+01  9.962178e+01  ...  1.084868e+02  1.071845e+02   
...                        ...           ...  ...           ...           ...   
Volume    YUM     1.497600e+06  1.163500e+06  ...  2.263500e+06  2.834400e+06   
          ZBH     1.180071e+06  1.451476e+06  ...  1.896500e+06  1.985600e+06   
          ZBRA    2.472000e+05  3.058000e+05  ...  3.666000e+05  3.779000e+05   
          ZTS     2.213900e+06  2.516500e+06  ...  3.018300e+06  2.728300e+06   
          ^GSPC   5.198480e+09  5.369820e+09  ...  4.544500e+09  5.246960e+09   

Date                2024-12-19    2024-12-20    2024-12-23    2024-12-24  \
Price     Ticker                                                           
Adj Close A       1.311343e+02  1.324339e+02  1.325225e+02  1.339994e+02   
          AAPL    2.479903e+02  2.526565e+02  2.534308e+02  2.563398e+02   
          ABBV    1.609095e+02  1.647183e+02  1.673545e+02  1.688649e+02   
          ABNB    1.293500e+02  1.342100e+02  1.345200e+02  1.349900e+02   
          ABT     1.076537e+02  1.093870e+02  1.094636e+02  1.098945e+02   
...                        ...           ...           ...           ...   
Volume    YUM     1.939200e+06  3.212900e+06  1.479100e+06  5.330000e+05   
          ZBH     2.672500e+06  3.969100e+06  1.948300e+06  4.586000e+05   
          ZBRA    4.323000e+05  8.264000e+05  2.383000e+05  8.870000e+04   
          ZTS     3.713700e+06  6.220100e+06  2.437800e+06  1.023600e+06   
          ^GSPC   4.896880e+09  8.223220e+09  3.593280e+

## 1.2 Data Validation and alignment

In [8]:
# Check datatypes of the columns
print(type(market_data_1.columns))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [9]:
# Drop rows that are entirely null
print('Before cleaning:', market_data_1.shape)
market_data_1 = market_data_1.dropna(how='all')
print('After cleaning:', market_data_1.shape)

Before cleaning: (3024, 1005)
After cleaning: (2994, 1005)


In [10]:
# Check for duplicate dates in the columns
duplicate_dates = market_data_1.columns[market_data_1.columns.duplicated()]
print(f'Number of duplicate dates: {len(duplicate_dates)}')

Number of duplicate dates: 0


In [11]:
# Check for Non-positive Prices (<= 0)
print('Non positive prices (<= 0)')
for price in ['Open', 'High', 'Low', 'Close', 'Adj Close']:
    non_pos_count = (market_data_1.xs(price, level='Price') <= 0).sum().sum()
    print(f'{price} {non_pos_count}')
print()

Non positive prices (<= 0)
Open 0
High 0
Low 0
Close 0
Adj Close 0



In [12]:
# Check for Volume Errors (< 0 or == 0)
print('Volume errors')
volume_data = market_data_1.xs('Volume', level='Price')
neg_volume = (volume_data < 0).sum().sum()
zero_volume = (volume_data == 0).sum().sum()
print(f'Negative Volume counts: {neg_volume}')
print(f'Zero Volume counts:     {zero_volume}')
print()

Volume errors
Negative Volume counts: 0
Zero Volume counts:     452



In [13]:
# Check for Unequal Trading Histories
print('Unequal trading histories')
adj_close_data = market_data_1.xs('Adj Close', level='Price')
max_days = adj_close_data.shape[1]
trading_days_per_ticker = adj_close_data.notna().sum(axis=1)
shorter_histories = trading_days_per_ticker[trading_days_per_ticker < max_days]
print(f'Total tickers with full history ({max_days} days): {len(trading_days_per_ticker) - len(shorter_histories)}')
print(f'Tickers with shorter histories: {len(shorter_histories)}')
if len(shorter_histories) > 0:
    print("Tickers with shorter histories:")
    print(shorter_histories.sort_values().head(16))
print()

Unequal trading histories
Total tickers with full history (1005 days): 489
Tickers with shorter histories: 10
Tickers with shorter histories:
Ticker
GEV     193
SOLV    194
RDDT    197
VLTO    313
GEHC    513
CEG     742
HOOD    862
APP     935
COIN    936
EXE     979
dtype: int64



In [14]:
# Check for Corporate Actions (Close not equal to Adj Close)
print('Corporate actions')
close_data = market_data_1.xs('Close', level='Price')
adj_close_data = market_data_1.xs('Adj Close', level='Price')
adjusted_mask = (close_data != adj_close_data).any(axis=1)
adjusted_tickers_count = adjusted_mask.sum()
print(f'Number of tickers with coporate actions: {adjusted_tickers_count}')

Corporate actions
Number of tickers with coporate actions: 410


In [15]:
# Identify tickers with unequal histories
unequal_tickers = shorter_histories.index.tolist()

# Drop them from market_data_1 along the 'Ticker' level of the MultiIndex
print('Shape before removing:', market_data_1.shape)
market_data_1 = market_data_1.drop(index=unequal_tickers, level='Ticker')
print('Shape after removing: ', market_data_1.shape)

Shape before removing: (2994, 1005)
Shape after removing:  (2934, 1005)


In [16]:
# Check for missing values
print('Missing values (NaN counts) per Price type')
price_types = ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
for price in price_types:
    nan_count = market_data_1.xs(price, level='Price').isna().sum().sum()
    print(f'{price} {nan_count}')
print()

Missing values (NaN counts) per Price type
Adj Close 0
Close 0
High 0
Low 0
Open 0
Volume 0



In [17]:
# Split the transposed dataset into separate DataFrames by Price type
adj_close_df = market_data_1.xs('Adj Close', level='Price')
close_df = market_data_1.xs('Close', level='Price')
high_df = market_data_1.xs('High', level='Price')
low_df = market_data_1.xs('Low', level='Price')
open_df = market_data_1.xs('Open', level='Price')
volume_df = market_data_1.xs('Volume', level='Price')

# Print the shape of each split dataset
print('Adj Close Shape:', adj_close_df.shape)
print('Close Shape:    ', close_df.shape)
print('High Shape:     ', high_df.shape)
print('Low Shape:      ', low_df.shape)
print('Open Shape:     ', open_df.shape)
print('Volume Shape:   ', volume_df.shape)

# Display a preview of the Adjusted Close DataFrame
display(adj_close_df.head())

Adj Close Shape: (489, 1005)
Close Shape:     (489, 1005)
High Shape:      (489, 1005)
Low Shape:       (489, 1005)
Open Shape:      (489, 1005)
Volume Shape:    (489, 1005)


Date,2021-01-04,2021-01-05,2021-01-06,2021-01-07,2021-01-08,2021-01-11,2021-01-12,2021-01-13,2021-01-14,2021-01-15,...,2024-12-17,2024-12-18,2024-12-19,2024-12-20,2024-12-23,2024-12-24,2024-12-26,2024-12-27,2024-12-30,2024-12-31
Ticker,,,,,,,,,,,,,,,,,,,,,
A,114.009125,114.941254,118.093231,121.235596,122.100471,123.147919,122.013962,122.023582,121.110672,121.283638,...,134.353806,130.957031,131.134262,132.433884,132.522522,133.999359,133.733536,133.447983,132.345245,132.511002
AAPL,125.632492,127.185806,122.904518,127.098434,128.195435,125.215042,125.040314,127.069305,125.147087,123.428757,...,251.653732,246.262894,247.990326,252.656494,253.430832,256.339783,257.153839,253.748550,250.382950,248.615799
ABBV,84.219147,85.090027,84.354965,85.257790,85.705208,87.103416,87.710617,89.843857,90.231850,89.334625,...,164.530716,162.316727,160.909470,164.718338,167.354507,168.864899,168.114380,166.998001,165.299973,166.707184
ABNB,139.149994,148.300003,142.770004,151.270004,149.770004,148.130005,160.800003,169.990005,180.500000,169.270004,...,132.240005,127.540001,129.350006,134.210007,134.520004,134.990005,135.320007,133.384995,131.809998,131.410004
ABT,97.269699,98.473183,98.268173,99.222069,99.498421,98.811951,97.029007,99.979813,98.493980,99.621780,...,108.486847,107.184502,107.653732,109.387001,109.463608,109.894531,110.382904,110.114777,108.017624,108.314476


## 1.3 Weekly Return and Weekly Beta

### 1.3.1 Mean weekly return

In [18]:
# Define weekly returns calculation fucntion

def calculate_weekly_returns(adj_close_df, shares=None):
    """Calculate weekly returns from adj_close_df"""
    prices = adj_close_df if shares is None else adj_close_df.loc[shares]

    prices = prices.T.copy()
    prices.index = pd.to_datetime(prices.index)
    prices = prices.sort_index()

    weekly_prices = prices.resample("W-FRI").last()
    return weekly_prices.pct_change(fill_method=None)

In [19]:
# Calculate weekly returns
weekly_returns = calculate_weekly_returns(adj_close_df)
print('Shape:', weekly_returns.shape)
display(weekly_returns)

Shape: (209, 489)


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS,^GSPC
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-15,-0.006690,-0.037183,0.042348,0.130200,0.001240,-0.048024,-0.036505,-0.055700,0.006444,-0.007038,...,-0.025681,-0.012165,0.053454,-0.036134,-0.056741,-0.004946,0.007361,0.009249,-0.038962,-0.014754
2021-01-22,0.000951,0.093834,0.003076,0.074437,0.013836,-0.039781,0.004416,0.031348,-0.013756,0.011877,...,0.011484,-0.003079,-0.009605,-0.008695,-0.021383,0.004408,0.005558,-0.000929,0.012522,0.019431
2021-01-29,-0.048761,-0.051126,-0.075591,0.009677,0.095268,-0.057040,-0.050438,-0.028935,-0.053027,-0.053200,...,-0.073709,-0.011890,-0.054607,-0.058577,-0.031048,-0.052292,-0.045587,-0.051389,-0.055593,-0.033120
2021-02-05,0.025048,0.037924,0.060988,0.063606,0.003560,0.076090,0.048032,0.072694,0.009706,0.065787,...,0.175424,-0.008595,0.113961,0.016565,0.113076,0.037935,0.040476,0.049713,0.032609,0.046467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-06,0.018917,0.023217,-0.036845,0.005951,-0.026690,-0.028693,0.000911,0.071773,0.000688,-0.058242,...,-0.000106,-0.038864,-0.037216,-0.002998,0.109543,-0.004629,-0.036307,0.005700,0.006905,0.009597
2024-12-13,-0.011239,0.021784,-0.016006,-0.043602,-0.019204,-0.045078,-0.011084,-0.157823,-0.003084,0.019059,...,0.001695,-0.023229,-0.024038,-0.045660,-0.063817,-0.004577,-0.011941,-0.013999,0.009747,-0.006433
2024-12-20,-0.032302,0.025632,0.012747,0.024895,0.007497,-0.026012,0.021440,-0.039769,-0.022298,-0.036450,...,-0.059029,-0.010569,-0.044839,-0.028690,-0.025332,-0.033869,0.003560,-0.026140,-0.074868,-0.019871


### 1.3.2 Weekly beta

In [20]:
def calculate_weekly_beta(weekly_returns, market_returns):
    """
    Calculates weekly beta over the full study period for fully aligned datasets.
    """
    # 1. Exclude incomplete first and final calendar weeks
    clean_weekly_returns = weekly_returns.iloc[1:-1]
    clean_market_returns = market_returns.iloc[1:-1]

    # Ensure no infinite values
    clean_weekly_returns = clean_weekly_returns.replace([np.inf, -np.inf], np.nan)
    clean_market_returns = clean_market_returns.replace([np.inf, -np.inf], np.nan)

    # 2. Market benchmark variance over the full study period
    market_var = clean_market_returns.var(ddof=1)
    if market_var <= 0:
        raise ValueError("Market return variance must be greater than zero.")

    results = []
    total_aligned_weeks = len(clean_market_returns)

    # 3. Compute beta across the common aligned series
    for share in clean_weekly_returns.columns:
        paired = pd.concat([clean_weekly_returns[share], clean_market_returns], axis=1).dropna()

        # Check for 100% complete alignment across the full study period
        if len(paired) == total_aligned_weeks:
            cov = paired.iloc[:, 0].cov(paired.iloc[:, 1])
            beta = cov / market_var
        else:
            beta = np.nan

        results.append({
            "Share": share,
            "Beta": beta,
            "Paired_weeks": len(paired)
        })

    return pd.DataFrame(results).set_index("Share")

In [21]:
# Extract market weekly returns (^GSPC) from the overall weekly_returns DataFrame
market_weekly_returns = weekly_returns['^GSPC']

# Drop the benchmark from the individual stocks' returns
stock_weekly_returns = weekly_returns.drop(columns=['^GSPC'])

print("Market weekly returns extracted. Shape:", market_weekly_returns.shape)
print("Stocks weekly returns shape:", stock_weekly_returns.shape)

Market weekly returns extracted. Shape: (209,)
Stocks weekly returns shape: (209, 488)


In [22]:
# Calculate weekly betas
weekly_betas_df = calculate_weekly_beta(stock_weekly_returns, market_weekly_returns)

print("Weekly Betas shape:", weekly_betas_df.shape)
display(weekly_betas_df.head())

Weekly Betas shape: (488, 2)


,Beta,Paired_weeks
Share,,
A,1.074724,207
AAPL,1.174836,207
ABBV,0.425366,207
ABNB,1.472709,207
ABT,0.583544,207


In [23]:
# Combine into one DataFrame
combined_df = (
    stock_weekly_returns
    .rename_axis(index="Week", columns="Share")
    .reset_index()
    .melt(
        id_vars="Week",
        var_name="Share",
        value_name="Weekly_Return"
    )
    .join(weekly_betas_df[["Beta"]], on="Share")
    .sort_values(["Share", "Week"])
    .reset_index(drop=True)
)

combined_df

,Week,Share,Weekly_Return,Beta
0,2021-01-08,A,NaN,1.074724
1,2021-01-15,A,-0.006690,1.074724
2,2021-01-22,A,0.000951,1.074724
3,2021-01-29,A,-0.048761,1.074724
4,2021-02-05,A,0.025048,1.074724
...,...,...,...,...
101987,2024-12-06,ZTS,0.006905,0.979783
101988,2024-12-13,ZTS,0.009747,0.979783
101989,2024-12-20,ZTS,-0.074868,0.979783
101990,2024-12-27,ZTS,-0.001456,0.979783


In [24]:
# Count null values in each column
print("Null values per column:")
print(combined_df.isna().sum())

# Remove rows with any null values
rows_before = len(combined_df)
combined_df = combined_df.dropna().reset_index(drop=True)

print("Rows removed:", rows_before - len(combined_df))
print("Rows remaining:", len(combined_df))

Null values per column:
Week               0
Share              0
Weekly_Return    488
Beta               0
dtype: int64
Rows removed: 488
Rows remaining: 101504


## 1.4 Adapted RFM Calculation

In [32]:
# Set the final common completed trading date as the obervation date
observation_date = adj_close_df.columns.max()
print('Observation Date:', observation_date)

Observation Date: 2024-12-31 00:00:00


In [35]:
# Get the volume for last 60 days for Recency and Frequency claulation
volume_60_days = volume_df.loc[:, :observation_date].iloc[:, -60:]
print(f"Volume_60_day dataframe shape: {volume_60_days.shape}")
display(volume_60_days.head())

Volume_60_day dataframe shape: (489, 60)


Date,2024-10-07,2024-10-08,2024-10-09,2024-10-10,2024-10-11,2024-10-14,2024-10-15,2024-10-16,2024-10-17,2024-10-18,...,2024-12-17,2024-12-18,2024-12-19,2024-12-20,2024-12-23,2024-12-24,2024-12-26,2024-12-27,2024-12-30,2024-12-31
Ticker,,,,,,,,,,,,,,,,,,,,,
A,836700.0,915000.0,634700.0,1288900.0,1427400.0,1765200.0,1251200.0,1702100.0,1592900.0,1185200.0,...,2430600.0,1746900.0,2166500.0,4363900.0,1194900.0,370200.0,556600.0,631800.0,993600.0,719600.0
AAPL,39505400.0,31855700.0,33591100.0,28183500.0,31759200.0,39882100.0,64751400.0,34082200.0,32993800.0,46431500.0,...,51356400.0,56774100.0,60882300.0,147495300.0,40858800.0,23234700.0,27237100.0,42355300.0,35557500.0,39480700.0
ABBV,5033800.0,4419000.0,4478900.0,3562500.0,2973000.0,3623900.0,6553500.0,5576000.0,4419000.0,5607900.0,...,7664700.0,5471600.0,6110500.0,18950900.0,6945600.0,2692800.0,2759300.0,2745400.0,3405700.0,3326400.0
ABNB,2847800.0,3304400.0,3731800.0,3060000.0,2714700.0,2314000.0,3502700.0,3691500.0,3814000.0,2391100.0,...,3125800.0,4236700.0,4744000.0,15471700.0,4728300.0,1677100.0,2067000.0,2650100.0,2458800.0,2312300.0
ABT,6912900.0,4341500.0,4256900.0,4068300.0,3471300.0,3674100.0,6544900.0,7356200.0,5046900.0,6741800.0,...,5805600.0,6454600.0,7738700.0,12311900.0,4028200.0,1480000.0,2960500.0,2829500.0,3475800.0,3519900.0


In [36]:
# Get the volume for last 60 days for Recency and Frequency claulation
close_60_days = close_df.loc[:, :observation_date].iloc[:, -60:]
print(f"Close_60_day dataframe shape: {close_60_days.shape}")
display(close_60_days.head())

Close_60_day dataframe shape: (489, 60)


Date,2024-10-07,2024-10-08,2024-10-09,2024-10-10,2024-10-11,2024-10-14,2024-10-15,2024-10-16,2024-10-17,2024-10-18,...,2024-12-17,2024-12-18,2024-12-19,2024-12-20,2024-12-23,2024-12-24,2024-12-26,2024-12-27,2024-12-30,2024-12-31
Ticker,,,,,,,,,,,,,,,,,,,,,
A,143.889999,144.610001,144.800003,142.740005,143.820007,144.250000,144.580002,139.110001,137.419998,138.830002,...,136.460007,133.009995,133.190002,134.509995,134.600006,136.100006,135.830002,135.539993,134.419998,134.339996
AAPL,221.690002,225.770004,229.539993,229.039993,227.550003,231.300003,233.850006,231.779999,232.149994,235.000000,...,253.479996,248.050003,249.789993,254.490005,255.270004,258.200012,259.019989,255.589996,252.199997,250.419998
ABBV,193.610001,193.009995,194.750000,194.350006,194.190002,195.649994,191.860001,190.460007,188.570007,188.860001,...,175.380005,173.020004,171.520004,175.580002,178.389999,180.000000,179.199997,178.009995,176.199997,177.699997
ABNB,130.229996,132.500000,135.490005,132.809998,134.399994,134.789993,133.270004,135.149994,136.020004,136.449997,...,132.240005,127.540001,129.350006,134.210007,134.520004,134.990005,135.320007,133.384995,131.809998,131.410004
ABT,113.360001,114.629997,115.949997,115.620003,116.120003,117.250000,116.050003,117.820000,117.889999,119.389999,...,113.290001,111.930000,112.419998,114.230003,114.309998,114.760002,115.269997,114.989998,112.800003,113.110001


In [42]:
# Define the unusually high volume day
# Calculate the rolling 20-session mean volume excluding the current day
rolling_20_mean = volume_df.shift(1).rolling(window=20).mean()

# Calculate the rolling threshold (twice the 20-session mean)
volume_threshold = 2.0 * rolling_20_mean

# Flag unusually high volume days (True where daily volume > threshold)
high_vol_days = (volume_df > volume_threshold)
print("High volume days identified")

High volume days identified


### 1.4.1 Recency

In [44]:
# Ensure observation_date is a pandas Timestamp
observation_date = pd.to_datetime(observation_date)

# Filter high volume days up to the observation date
high_vol_study_period = high_vol_days.loc[:, :observation_date]

# Initialize series for recency
recency_series = pd.Series(index=high_vol_study_period.index, dtype=float)

for ticker in high_vol_study_period.index:
    # Find dates where this ticker had high volume days
    ticker_series = high_vol_study_period.loc[ticker]
    high_vol_dates = ticker_series.index[ticker_series == True]

    if len(high_vol_dates) > 0:
        # Find the latest qualifying high volume date
        latest_date = pd.to_datetime(high_vol_dates.max())
        # Calculate calendar day difference
        days_diff = (observation_date - latest_date).days
        recency_series[ticker] = days_diff
    else:
        # No qualifying high volume event found
        recency_series[ticker] = np.nan

# Convert to a DataFrame
recency_df = recency_series.to_frame(name='Recency')

print("Recency calculated successfully.")
print(f"Missing values (NaNs) in Recency: {recency_df['Recency'].isna().sum()}")
display(recency_df.head(10))

Recency calculated successfully.
Missing values (NaNs) in Recency: 191


,Recency
Ticker,
A,NaN
AAPL,NaN
ABBV,NaN
ABNB,NaN
ABT,NaN
ACGL,NaN
ACN,NaN
ADBE,NaN
ADI,NaN
